#### Imports

In [26]:
from pydantic import BaseModel, Field
from typing import Literal
from typing import Callable
import numpy as np
import os
import json
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
from huggingface_hub import InferenceClient
from sklearn.metrics.pairwise import cosine_similarity
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [27]:
class Intent(BaseModel):
    type: Literal["general", "out_of_scope", "asking_mental_health_question"] = Field(
        ...,
        description="general: for greeting, goodbye, thanks only; out_of_scope: for queries that are not related to mental health asking_mental_health_question: for queries that are related to mental health questions",
    )
    confidence: float = Field(
        ...,
        description="confidence score of the intent classification, between 0 and 1",
    )
    classifier: Literal["embedding", "llm"] = Field(
        ...,
        description="the classifier used for intent classification: embedding: a classifier based on sentence embeddings and cosine similarity; llm: a classifier based on a large language model",
    )

In [30]:


class IntentClassifier:
    def __init__(self, llm_fallback: Callable[[str], Intent] | None = None):
        self.llm_fallback = llm_fallback or self._ollama_llm_fallback
        self.ollama_host = os.getenv("OLLAMA_HOST")
        self.ollama_model = os.getenv("OLLAMA_MODEL")
        self.ollama_timeout_seconds = float(os.getenv("OLLAMA_TIMEOUT_SECONDS"))

        self.embedding_examples = {
            "general": [
                "hello",
                "thanks",
                "goodbye",
                "hi there",
                "thank you",
            ],
            "out_of_scope": [
                "what is the weather like",
                "tell me a joke",
                "what is the latest news",
                "what sports scores are",
            ],
            "asking_mental_health_question": [
                "I feel anxious and need help",
                "I am depressed",
                "can you help me with my stress",
                "I need therapy advice",
            ],
        }

        self.embedding_threshold = 0.65
        self.embedding_model = "BAAI/bge-small-en-v1.5"
        self.embedding_client = InferenceClient(
            provider="hf-inference",
            api_key=os.environ["HF_TOKEN"],
        )

        self.embedding_examples_embeddings = {
            intent_type: self._normalize_embeddings(
                np.asarray(
                    self.embedding_client.feature_extraction(
                        examples,
                        model=self.embedding_model,
                    ),
                    dtype=np.float32,
                )
            )
            for intent_type, examples in self.embedding_examples.items()
        }

    def classify(self, text: str) -> Intent:
        embedding_intent = self._embedding_classifier(text)
        if embedding_intent is not None:
            return embedding_intent

        return self.llm_fallback(text)

    def _embedding_classifier(self, text: str) -> Intent | None:
        text_embedding = self._get_embedding(text)
        best_intent = None
        best_score = 0.0

        for intent_type, example_embeddings in self.embedding_examples_embeddings.items():
            if example_embeddings.size == 0:
                continue
            scores = example_embeddings @ text_embedding
            max_score = float(np.max(scores))
            if max_score > best_score:
                best_score = max_score
                best_intent = intent_type

        if best_intent is not None and best_score >= self.embedding_threshold:
            return Intent(
                type=best_intent,
                confidence=best_score,
                classifier="embedding",
            )
        return None

    def _ollama_llm_fallback(self, text: str) -> Intent:
        payload = {
            "model": self.ollama_model,
            "messages": [
                {"role": "system", "content": self._system_prompt()},
                {"role": "user", "content": text},
            ],
            "stream": False,
            "format": "json",
            "options": {
                "temperature": 0,
            },
        }

        request = Request(
            f"{self.ollama_host.rstrip('/')}/api/chat",
            data=json.dumps(payload).encode("utf-8"),
            headers={"Content-Type": "application/json"},
            method="POST",
        )

        try:
            with urlopen(request, timeout=self.ollama_timeout_seconds) as response:
                body = json.loads(response.read().decode("utf-8"))
            message = body.get("message", {})
            content = message.get("content", "")
            return self._parse_llm_intent(content)
        except (HTTPError, URLError, TimeoutError, json.JSONDecodeError, KeyError, ValueError):
            return self._heuristic_llm_fallback(text)

    def _heuristic_llm_fallback(self, text: str) -> Intent:
        normalized = text.lower()
        if any(token in normalized for token in ["help", "therapy", "anxi", "depress", "suicid", "panic", "stress", "lonely"]):
            return Intent(type="asking_mental_health_question", confidence=0.72, classifier="llm")

        if any(token in normalized for token in ["weather", "joke", "news", "movie", "sports", "stock", "time"]):
            return Intent(type="out_of_scope", confidence=0.68, classifier="llm")

        return Intent(type="general", confidence=0.55, classifier="llm")

    def _parse_llm_intent(self, content: str) -> Intent:
        cleaned = str(content).strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.strip("`")
            cleaned = cleaned.replace("json\n", "", 1).strip()

        start = cleaned.find("{")
        end = cleaned.rfind("}")
        if start != -1 and end != -1 and end > start:
            cleaned = cleaned[start : end + 1]

        data = json.loads(cleaned)
        intent_type = data.get("type", "general")
        if intent_type not in {"general", "out_of_scope", "asking_mental_health_question"}:
            intent_type = "general"

        confidence = data.get("confidence", 0.5)
        try:
            confidence = float(confidence)
        except (TypeError, ValueError):
            confidence = 0.65
        confidence = max(0.0, min(1.0, confidence))

        return Intent(
            type=intent_type,
            confidence=confidence,
            classifier="llm",
        )

    def _system_prompt(self) -> str:
        return (
            "You are a strict intent classification engine for a mental-health support assistant. "
            "Classify the user's message into exactly one label: general, out_of_scope, or asking_mental_health_question. "
            "Use general only for greetings, goodbyes, and thanks. "
            "Use asking_mental_health_question for any mental-health-related question, emotional distress, therapy, anxiety, depression, panic, stress, loneliness, self-harm, suicidal thoughts, mood, or requests for support about wellbeing. "
            "Use out_of_scope for everything else that is not about mental health. "
            "Return only valid JSON with exactly these keys: type, confidence, classifier. "
            "The classifier value must always be llm. "
            "Confidence must be a number from 0 to 1 reflecting how sure you are. "
            "Do not include markdown, code fences, commentary, or extra keys."
        )

    def _get_embedding(self, text: str) -> np.ndarray:
        embedding = self.embedding_client.feature_extraction(
            text,
            model=self.embedding_model,
        )
        return self._normalize_embeddings(np.asarray(embedding, dtype=np.float32))

    @staticmethod
    def _normalize_embeddings(embeddings: np.ndarray) -> np.ndarray:
        embeddings = np.asarray(embeddings, dtype=np.float32)
        if embeddings.ndim == 1:
            norm = np.linalg.norm(embeddings)
            return embeddings / norm if norm > 0 else embeddings
        norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        return embeddings / norms


In [31]:
# Test the IntentClassifier
classifier = IntentClassifier()

# Test cases
test_cases = [
    ("Hello there!", "general"),
    # ("Thank you so much", "general"),
    # ("What's the weather today?", "out_of_scope"),
    # ("Tell me a joke", "out_of_scope"),
    # ("I'm feeling anxious and stressed", "asking_mental_health_question"),
    # ("I need help with my depression", "asking_mental_health_question"),
    # ("Can you recommend a therapist?", "asking_mental_health_question"),
    # ("How is going on?", "general"),
    # ("What time is it?", "out_of_scope"),
    # ("I'm so lonely and need someone to talk to", "asking_mental_health_question"),
    # ("I feel depressed every time i go to study, help me to get off", "asking_mental_health_question")
    
    # write an example to trigger the llm model, write a very long text 
    
    ("I’m trying to decide how to organize a large collection of handwritten notes, scanned articles, meeting summaries, and personal reminders into a system that is easy to search later without becoming so complicated that I stop using it. I have a laptop, a phone, a few cloud folders, and a paper notebook, and right now the same idea is often stored in more than one place. I want a method that helps me decide where each new note should go, how to name it, when to merge duplicates, and how to find things again after a few weeks have passed. I also want the system to be simple enough that I can maintain it on busy days, but detailed enough that I do not lose useful context. If you were designing a practical workflow for this, what structure would you recommend, and how would you keep it from becoming messy over time?", "out_of_scope"),
    ("I want to know about the stock market", "out_of_scope"),
    
]

print("Testing IntentClassifier:\n")
for text, expected_type in test_cases:
    result = classifier.classify(text)
    match = "✓" if result.type == expected_type else "✗"
    print(f"{match} Input: '{text}'")
    print(f"  Expected: {expected_type}, Got: {result.type}")
    print(f"  Confidence: {result.confidence:.2f}, Classifier: {result.classifier}\n")

Testing IntentClassifier:

✓ Input: 'Hello there!'
  Expected: general, Got: general
  Confidence: 0.90, Classifier: embedding

✓ Input: 'I’m trying to decide how to organize a large collection of handwritten notes, scanned articles, meeting summaries, and personal reminders into a system that is easy to search later without becoming so complicated that I stop using it. I have a laptop, a phone, a few cloud folders, and a paper notebook, and right now the same idea is often stored in more than one place. I want a method that helps me decide where each new note should go, how to name it, when to merge duplicates, and how to find things again after a few weeks have passed. I also want the system to be simple enough that I can maintain it on busy days, but detailed enough that I do not lose useful context. If you were designing a practical workflow for this, what structure would you recommend, and how would you keep it from becoming messy over time?'
  Expected: out_of_scope, Got: out_of_